[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml/corrections/seance1_correction.ipynb)

# Séance 4.1 — Le Machine Learning : prédire n'est pas expliquer

**Correction** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- dire ce qui sépare un modèle qui explique d'un modèle qui prédit
- découper un jeu de données en apprentissage et test, et dire pourquoi
- mesurer une erreur de prédiction en euros avec la RMSE et la MAE
- reconnaître un surapprentissage à l'écart entre les deux jeux
- repérer une fuite de données — l'erreur qui donne un modèle parfait et inutile

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc4_ml/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")

X = cmd[["qte", "nart"]]   ## ce qu'on connait avant de facturer
y = cmd["ca"]              ## ce qu'on veut prevoir

# test_size=0.25 : un quart des lignes mis de cote pour la notation
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=67)
print(len(X_tr), "commandes d'apprentissage,", len(X_te), "de test")

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Ajuster et prédire

> **Votre mission :**
> - Ajuster une régression linéaire sur le jeu d'**apprentissage** → `m`.
> - Prédire sur le jeu de **test** → `p`.
> - Mettre la première prédiction, arrondie à 2 décimales, dans `p1`.

In [ ]:
# On apprend sur X_tr / y_tr, on predit sur X_te : jamais l'inverse
m = LinearRegression().fit(X_tr, y_tr)   ## apprendre
p = m.predict(X_te)                      ## noter, sur des lignes jamais vues

p1 = round(p[0], 2)   ## une prediction par ligne de test
print(len(p), "predictions | la premiere :", p1)

In [ ]:
verifier("1a - nombre de predictions", len(p) == 489, "on predit sur le jeu de test")
verifier("1b - premiere prediction", abs(p1 - 370.99) < 1,
         "ajustez sur X_tr et y_tr, puis predisez sur X_te")

### Exercice 2 — L'erreur en euros

> **Votre mission :**
> - Calculer la MAE et la RMSE du modèle **sur le jeu de test** → `mae` et `rmse`, arrondies à 1 décimale.
> - Rappel : la RMSE est la racine de `mean_squared_error`.

In [ ]:
mae = round(mean_absolute_error(y_te, p), 1)   ## en euros

# ** 0.5 : la racine carree. RMSE = Root Mean Squared Error
rmse = round(mean_squared_error(y_te, p) ** 0.5, 1)   ## punit les gros ecarts

print("MAE", mae, "euros | RMSE", rmse, "euros")

In [ ]:
verifier("2a - MAE", abs(mae - 217.5) < 1, "mean_absolute_error(y_te, p)")
verifier("2b - RMSE", abs(rmse - 583.3) < 1, "la racine carree s'ecrit ** 0.5")

### Exercice 3 — Les deux notes du même modèle

> **Votre mission :**
> - Calculer le R² du modèle **en apprentissage** → `r2_tr`, et **en test** → `r2_te`, arrondis à 3 décimales.
> - Lequel des deux est le meilleur ? De combien ? Cet écart est-il inquiétant ?

In [ ]:
r2_tr = round(r2_score(y_tr, m.predict(X_tr)), 3)   ## sur le vu
r2_te = round(r2_score(y_te, m.predict(X_te)), 3)   ## sur le jamais vu

print("apprentissage", r2_tr, "| test", r2_te)
print("ecart :", round(r2_tr - r2_te, 3))

# L'apprentissage est meilleur, comme toujours : les coefficients ont ete
# calcules pour coller a CES lignes-la. Quatre points d'ecart, c'est le
# prix normal a payer. Un ecart de quarante points, lui, s'appellerait du
# surapprentissage : le modele aurait retenu au lieu d'apprendre.

In [ ]:
verifier("3a - R2 en apprentissage", abs(r2_tr - 0.732) < 0.01, "predisez sur X_tr")
verifier("3b - R2 en test", abs(r2_te - 0.691) < 0.01, "predisez sur X_te")

### Exercice 4 — Lire ce que le modèle a appris

> **Votre mission :**
> - Un modèle ajusté, ce sont des **nombres** qu'on peut lire. Relever la constante → `constante`, puis les deux coefficients → `coef_qte` et `coef_nart` (2 décimales).
> - *Indice :* `m.intercept_` donne la constante, `m.coef_` les coefficients dans l'ordre des colonnes de `X`.
> - Puis, en commentaire : que prédit ce modèle pour une commande de **0 unité et 0 produit** ? Cette valeur a-t-elle un sens commercial ?

In [ ]:
constante = round(m.intercept_, 2)
coef_qte = round(m.coef_[0], 2)     ## premiere colonne de X : qte
coef_nart = round(m.coef_[1], 2)    ## deuxieme colonne : nart

print(f"ca = {constante} + {coef_qte} x qte + {coef_nart} x nart")

# 1,32 EUR par unite supplementaire, 3,64 EUR par produit distinct de plus.
# La constante vaut 62,86 : c'est ce que le modele predit pour une commande
# de zero unite et zero produit. Une telle commande n'existe pas — la
# constante POSITIONNE la droite, elle ne se lit pas comme un montant.

In [ ]:
verifier("4a - la constante", constante == 62.86, "m.intercept_, arrondi a 2 decimales")
verifier("4b - le coefficient de qte", coef_qte == 1.32, "m.coef_[0]")
verifier("4c - le coefficient de nart", coef_nart == 3.64, "m.coef_[1]")

### Exercice 5 — Pourquoi la RMSE dépasse toujours la MAE

> **Votre mission :**
> - En test, la MAE vaut 217,5 € et la RMSE 583,3 €. Calculer leur rapport → `rapport` (2 décimales).
> - Puis mesurer ce que pèsent les **5 % de commandes les plus mal prédites** dans le total des erreurs → `part_pires` (1 décimale, en %).
> - *Nouveau :* `serie.nlargest(k)` garde les `k` plus grandes valeurs.

In [ ]:
rapport = round(rmse / mae, 2)

ecarts = (y_te - p).abs()                    ## l'erreur de chaque commande
k = int(0.05 * len(ecarts))                  ## 5 % des commandes
part_pires = round(100 * ecarts.nlargest(k).sum() / ecarts.sum(), 1)

print("rapport RMSE/MAE :", rapport, "| part des 5 % pires :", part_pires, "%")

# La RMSE vaut 2,68 fois la MAE. Elle eleve les erreurs au CARRE avant de
# les moyenner : une erreur de 2 000 EUR y pese cent fois une erreur de
# 200 EUR. L'ecart entre les deux est donc un signal — et il se verifie :
# 5 % des commandes portent 44,3 % de l'erreur totale. La MAE seule
# cachait une poignee de devis tres faux.

In [ ]:
verifier("5a - le rapport RMSE/MAE", rapport == 2.68, "rmse divise par mae")
verifier("5b - le poids des 5 % pires", part_pires == 44.3,
         "nlargest(k) garde les k plus grandes erreurs")

### Exercice 6 — Prédire une commande qui arrive

> **Votre mission :**
> - Une commande arrive : **150 unités** (`qte`), **12 produits distincts** (`nart`).
> - Prédire son montant avec `m` → `devis`, arrondi à 2 décimales.
> - `predict` attend un tableau dont les colonnes portent les mêmes noms que `X`.

In [ ]:
# Les colonnes doivent porter les memes noms QUE DANS LE MEME ORDRE
nouvelle = pd.DataFrame({"qte": [150], "nart": [12]})   ## une seule ligne
devis = round(m.predict(nouvelle)[0], 2)   ## [0] : la premiere prediction

print("montant prevu :", devis, "euros")

In [ ]:
verifier("6 - montant prevu", abs(devis - 305.12) < 2,
         "la seconde colonne s'appelle nart")

### Exercice 7 — La fuite de données

> **Votre mission :**
> - Refaire un modèle en mettant `ca` parmi les variables explicatives → `r2_fuite`, arrondi à 3 décimales.
> - Le score est parfait. Expliquez en une phrase pourquoi ce modèle ne vaut rien.

In [ ]:
triche = cmd[["qte", "nart", "ca"]]   ## "ca" est la cible !
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(triche, y, test_size=0.25, random_state=67)

r2_fuite = round(r2_score(yf_te, LinearRegression().fit(Xf_tr, yf_tr).predict(Xf_te)), 3)
print(r2_fuite)

# On a donne la reponse au modele. Le jour ou une vraie commande arrive,
# la colonne ca n'existe pas encore : le modele est inutilisable.

In [ ]:
verifier("7 - modele parfait et inutile", r2_fuite == 1.0,
         "la colonne qui contient la reponse s'appelle ca")

### Exercice 8 — Question de synthèse

> **Votre mission :**
> - Le directeur logistique veut savoir s'il peut se fier au modèle pour dimensionner ses préparations.
> - Calculer l'erreur médiane absolue en euros → `err_med`, arrondie à 2 décimales.
> - La comparer à la médiane des commandes → `med_ca`. Que lui répondez-vous ?

In [ ]:
erreurs = (y_te - p).abs()

# La mediane des erreurs, plutot que leur moyenne : meme raisonnement
# qu'en seance 3.1, quelques grosses fautes tirent la moyenne
err_med = round(erreurs.median(), 2)   ## l'erreur d'une fois sur deux
med_ca = round(y_te.median(), 2)       ## a comparer a la commande typique
print("erreur mediane", err_med, "euros pour une commande mediane de", med_ca)

# Une reponse possible :
# "Sur une commande mediane de 343 EUR, nous nous trompons de 91 EUR une
#  fois sur deux, soit 27 %. C'est utilisable pour dimensionner un stock
#  global, pas pour engager un devis client."

In [ ]:
verifier("8a - erreur mediane", abs(err_med - 90.92) < 3, "la methode s'appelle median()")
verifier("8b - commande mediane", abs(med_ca - 342.90) < 3, "median() sur y_te")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 9 — Le découpage change-t-il la note ?

> **Votre mission :**
> - Refaire le découpage et l'évaluation avec cinq `random_state` différents (0, 1, 2, 3, 4).
> - Afficher le R² de test à chaque fois. De combien varie-t-il ?
> - Qu'en concluez-vous sur un score annoncé sans précision de découpage ?

In [ ]:
for graine in range(5):
    # Meme fichier, meme modele : seul le tirage du test change
    a, b, c, d = train_test_split(X, y, test_size=0.25, random_state=graine)
    r2 = r2_score(d, LinearRegression().fit(a, c).predict(b))
    print(f"random_state={graine} : R2 = {r2:.3f}")

# Le R2 va de 0,45 a 0,74 selon le tirage : annoncer "R2 = 0,69" sans dire
# sur quel decoupage, c'est annoncer un chiffre qu'on ne peut pas
# reproduire. La validation croisee de la seance 4.3 repond a ce probleme.

### Question 10 — Le modèle le plus bête

> **Votre mission :**
> - Construire un modèle de référence qui prédit **toujours la même valeur** : la moyenne des montants d'apprentissage.
> - Calculer sa RMSE de test. Quel R² obtient-il ?
> - Tout modèle doit battre celui-là, sinon il ne sert à rien.

In [ ]:
bete = np.full(len(y_te), y_tr.mean())   ## toujours la meme reponse

print("RMSE du modele bete :", round(mean_squared_error(y_te, bete) ** 0.5, 1))
print("RMSE du vrai modele :", round(mean_squared_error(y_te, p) ** 0.5, 1))
print("R2 du modele bete   :", round(r2_score(y_te, bete), 3))

# Le R2 du modele constant vaut zero a un millieme pres (RMSE 1 051 EUR,
# contre 583 pour le vrai modele) : c'est
# exactement ce que le R2 mesure — le gain par rapport a "toujours la
# moyenne". Ce modele de reference est le premier a construire, toujours.

### Question 11 — Prédire le prix moyen plutôt que le montant

> **Votre mission :**
> - Créer une cible `prix_moyen` = `ca` / `qte`, et tenter de la prédire à partir de `nart`.
> - Le R² de test est mauvais. Est-ce le modèle qui est mauvais, ou la question ?

In [ ]:
cible = cmd["ca"] / cmd["qte"]   ## le prix moyen d'un article
a, b, c, d = train_test_split(cmd[["nart"]], cible, test_size=0.25, random_state=67)

r2 = r2_score(d, LinearRegression().fit(a, c).predict(b))
print("R2 :", round(r2, 3))

# La question. Le nombre de produits distincts ne contient aucune information sur
# le prix unitaire moyen : un modele ne peut pas inventer un signal qui
# n'est pas dans les donnees. Un mauvais R2 est parfois une reponse.

### Question 12 — Un découpage qui triche

> **Votre mission :**
> - Les commandes sont triées par date. Refaire le découpage **sans mélanger** (`shuffle=False`) : le test devient les dernières commandes.
> - Comparer le R² de test à celui obtenu avec mélange.
> - Lequel des deux découpages imite le mieux la vraie vie ?

In [ ]:
# shuffle=False : le test devient les dernieres commandes, pas un tirage
a, b, c, d = train_test_split(X, y, test_size=0.25, shuffle=False)
r2_temps = r2_score(d, LinearRegression().fit(a, c).predict(b))

print("decoupage aleatoire :", round(r2_score(y_te, p), 3))
print("decoupage temporel  :", round(r2_temps, 3))

# Le decoupage temporel est le seul honnete ici : en vrai, on predit
# l'avenir avec le passe, jamais l'inverse. Un decoupage aleatoire laisse
# le modele apprendre sur des commandes POSTERIEURES a celles qu'il note.

### Question 13 — La fuite discrète

> **Votre mission :**
> - La fuite du cours était grossière. En voici une réaliste : ajouter une colonne `remise`, qui vaut 5 % du montant facturé.
> - Rien dans son nom ne dit qu'elle contient la réponse — et pourtant.
> - Mesurer le R² de test et expliquer ce qui s'est passé.

In [ ]:
discret = cmd[["qte", "nart"]].copy()
discret["remise"] = (cmd["ca"] * 0.05).round(2)   ## 5 % du montant facture

a, b, c, d = train_test_split(discret, y, test_size=0.25, random_state=67)
print("R2 :", round(r2_score(d, LinearRegression().fit(a, c).predict(b)), 3))

# R2 = 1,0. La remise vaut 5 % du montant : c'est la reponse, ecrite
# autrement. Aucune colonne ne s'appelait "ca", et pourtant la fuite est
# totale. Le test a appliquer : cette colonne serait-elle disponible AVANT
# que la commande soit facturee ? Non. Elle n'a rien a faire la.

### Question 14 — Choisir la bonne métrique

> **Votre mission :**
> - Comparer deux modèles : la régression linéaire `m`, et le **modèle nul** de la question 10 — celui qui prédit toujours la moyenne d'apprentissage.
> - Afficher la MAE et la RMSE de test des deux, puis le gain apporté par le modèle sur chacune.
> - Le gain est presque **deux fois plus grand sur la RMSE que sur la MAE**. Qu'est-ce que ça dit de ce que le modèle corrige ?
> - Et si l'entreprise craint surtout les **grosses** erreurs de devis, quelle métrique doit-elle regarder ?

In [ ]:
# Le modele nul : la moyenne d'apprentissage, repetee pour chaque commande
nul = np.full(len(y_te), y_tr.mean())

for nom, pred in {"modele nul": nul, "regression": p}.items():
    print(f"{nom:<12} MAE {mean_absolute_error(y_te, pred):7.1f}  "
          f"RMSE {mean_squared_error(y_te, pred) ** 0.5:7.1f}")

print(f"gain MAE  : {mean_absolute_error(y_te, nul) - mae:.1f} euros")
print(f"gain RMSE : {mean_squared_error(y_te, nul) ** 0.5 - rmse:.1f} euros")

# MAE 483,7 -> 217,5 : 266 EUR gagnes. RMSE 1 050,5 -> 583,3 : 467 EUR.
# Le gain est presque DOUBLE sur la RMSE, et ce n'est pas un hasard : la
# RMSE compte surtout les grosses erreurs, et c'est precisement la que le
# modele apporte le plus. Predire la moyenne se trompe enormement sur les
# tres grosses commandes ; la regression, elle, les suit.
#
# Consequence de gestion : une entreprise qui craint le devis catastrophique
# regarde la RMSE, et le modele lui apporte beaucoup. Une entreprise qui
# veut juste reduire l'erreur courante regarde la MAE, et le gain est plus
# modeste. Le choix de la metrique n'est pas technique, il est strategique.

### Question 15 — De quelle famille relève ce problème ?

> **Votre mission :**
> - Celle-ci ne demande pas de code. **Ajoutez une cellule de texte** sous celle-ci et classez les cinq situations ci-dessous dans les familles de la section 1 : apprentissage **supervisé** (régression ou classification), **non supervisé**, ou par **renforcement**.
> - 1. Estimer le loyer d'un appartement à partir de sa surface et de son quartier.
> - 2. Décider si une transaction par carte est frauduleuse ou non.
> - 3. Regrouper 40 000 clients en profils d'achat, sans catégorie définie à l'avance.
> - 4. Apprendre à un robot d'entrepôt à ranger des colis, en le récompensant quand il réussit.
> - 5. Prévoir le nombre de commandes de la semaine prochaine.
> - **Justifiez chaque réponse par une seule question :** *dispose-t-on d'un `y` connu à l'avance, et si oui, est-il un nombre ou une catégorie ?*

In [ ]:
# Rien a executer : la reponse est une cellule de TEXTE.
# Voici ce qu'il fallait y ecrire.
#
# 1. Loyer d'un appartement    -> SUPERVISE, regression.
#    On dispose de loyers connus, et y est un nombre. C'est exactement
#    la seance d'aujourd'hui, avec un autre fichier.
#
# 2. Transaction frauduleuse   -> SUPERVISE, classification.
#    On dispose de transactions deja etiquetees, et y vaut 0 ou 1.
#    C'est la seance 4.2, qui commence par exactement ce cas.
#
# 3. Regrouper 40 000 clients  -> NON SUPERVISE (clustering).
#    Il n'y a AUCUN y : personne n'a etiquete les clients a l'avance.
#    C'est la seance 4.4, et le mot "sans categorie definie a l'avance"
#    est ce qui le signale.
#
# 4. Robot d'entrepot          -> RENFORCEMENT.
#    Pas de bonnes reponses fournies, mais une recompense apres coup.
#    Le modele apprend par essais et erreurs repetes. C'est la seule
#    famille que ce cours ne pratiquera pas.
#
# 5. Commandes de la semaine   -> SUPERVISE, regression.
#    y est un nombre, et l'historique fournit des exemples etiquetes.
#
# La question qui tranche est toujours la meme, et elle tient en deux temps :
#   - existe-t-il un y connu ?  non -> non supervise (ou renforcement,
#     si une recompense arrive apres l'action) ;
#   - si oui, ce y est-il un NOMBRE (regression) ou une CATEGORIE
#     (classification) ?
#
# Remarquez que trois des cinq situations relevent de la meme famille que
# la seance d'aujourd'hui. C'est le cas le plus frequent en entreprise.

### Question 16 — Question de synthèse

> **Votre mission :**
> - On vous demande une note d'une page : *« peut-on automatiser le devis à partir du nombre d'unités et de produits distincts ? »*
> - Produire les trois chiffres qui fondent votre réponse : l'erreur typique en euros, sa part du montant médian, et la part des commandes prédites à moins de 20 % près.
> - Puis rédigez la recommandation en commentaire.

In [ ]:
erreurs = (y_te - p).abs()
ecart_relatif = erreurs / y_te   ## une erreur de 95 EUR ne pese pas pareil

print("erreur mediane      :", round(erreurs.median(), 2), "euros")
print("soit                :", round(100 * (erreurs.median() / y_te.median()), 1), "%")
print("commandes a +/- 20 %:", round(100 * (ecart_relatif < 0.20).mean(), 1), "%")

# Recommandation possible :
# "Le modele predit deux commandes sur cinq a moins de 20 % pres, et se
#  trompe de 91 EUR sur un panier median de 343 EUR. C'est suffisant pour
#  dimensionner un stock hebdomadaire, insuffisant pour engager un devis
#  client. Nous recommandons un usage interne uniquement, et la collecte
#  du type d'article — la variable qui manque visiblement au modele."